# Zero-Shot Image Segmentation with CLIPSeg

This notebook uses the `CIDAS/clipseg-rd64-refined` model to perform image segmentation on the images in the `images` folder based on text prompts.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

%matplotlib inline

## 1. Load CLIPSeg Model and Processor

In [ ]:
model_checkpoint = "CIDAS/clipseg-rd64-refined"
processor = CLIPSegProcessor.from_pretrained(model_checkpoint)
model = CLIPSegForImageSegmentation.from_pretrained(model_checkpoint)
model.eval()

print(f"Model '{model_checkpoint}' loaded successfully.")

## 2. Segmentation and Visualization Function

In [ ]:
def segment_image(image_path, prompts=["car", "tree", "sky", "road"]):
    """Performs segmentation based on text prompts."""
    image = Image.open(image_path).convert("RGB")
    inputs = processor(text=prompts, images=[image] * len(prompts), padding="max_length", return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predicted masks
    preds = torch.sigmoid(outputs.logits)
    
    # Visualizing the results in a grid
    n_prompts = len(prompts)
    fig, axes = plt.subplots(1, n_prompts + 1, figsize=(4 * (n_prompts + 1), 5), constrained_layout=True)
    
    # Show original image
    axes[0].imshow(image)
    axes[0].set_title(f"IMAGE: {os.path.basename(image_path)}", fontweight='bold', fontsize=12)
    axes[0].axis("off")
    
    # Show each generated heatmap/mask
    for i, prompt in enumerate(prompts):
        ax = axes[i + 1]
        ax.imshow(preds[i].cpu().detach())
        ax.set_title(f"Mask for: '{prompt}'", fontweight='bold', fontsize=12)
        ax.axis("off")
    
    plt.show()

## 3. Run Inference on Images

In [ ]:
images_dir = 'images'
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# EDIT THIS LIST TO CHANGE THE SEGMENTATION TARGETS
my_prompts = ["parrot", "cat", "dog", "person", "sheep"]

if not image_files:
    print(f"No images found in {images_dir}.")
else:
    for filename in image_files[:5]: # Proccessing first 5 images
        image_path = os.path.join(images_dir, filename)
        print(f"Segmenting {filename}...")
        segment_image(image_path, prompts=my_prompts)